# Получение топ-250 фильмов 

### Условие задачи

С API необходимо собрать данные о топ-250 фильмах. Набор данных должен включать название, год выпуска, рейтинг и количество оценок. Нужно извлечь данные со страницы и сохранить их в CSV файл.

### Подключение библиотек

In [19]:
from bs4 import BeautifulSoup as bs
import requests
import pandas as pd
import time

### Получение информаций

In [20]:
base_url = 'https://premier.one'
sub_url = '/collections/top-250-filmov-na-premier?page='
result_list = {'title': [], 'genre': [], 'year': [],
               'country': [], 'rate': [], 'description': []}

In [21]:
get_top_250_films = 0

In [22]:

for i in range(1, 6):
    url = base_url + sub_url + str(i)
    page = requests.get(url)
    page.status_code
    soup = bs(page.text, 'html.parser')
    links = soup.find_all('a', class_="e-poster")
    for link in links:
        try:
            url = base_url + link.get('href')
            page = requests.get(url)
            inner_soup = bs(page.text, 'html.parser')

            t_el = inner_soup.find('div', class_='w-show-promo-new__title-small')
            y_el = inner_soup.find('a', class_='w-show-promo-new__year-link')
            c_el = inner_soup.find('a', class_='w-show-promo-new__country-link')
            r_el = inner_soup.find('span', class_='a-badge-rating__value')
            
            d_v = inner_soup.find('span', class_='a-cropped-text__visible-text')
            d_h = inner_soup.find('span', class_='a-cropped-text__hide')
            full_desc = (d_v.text.strip() if d_v else "") + (d_h.text.strip() if d_h else "")
    
            result_list['title'].append(t_el.text.strip() if t_el else "Без названия")
            result_list['year'].append(y_el.text.strip() if y_el else "—")
            result_list['country'].append(c_el.text.strip() if c_el else "—")
            result_list['rate'].append(r_el.text.strip() if r_el else "0.0")
            result_list['description'].append(full_desc if full_desc else "Нет описания")
            
            g_els = inner_soup.find_all('a', class_='w-show-promo-new__genre-link')
            result_list['genre'].append(", ".join([i.text.strip() for i in g_els]) if g_els else "—")
            
            get_top_250_films += 1
    
        except Exception as e:
            print(f"Ошибка на {url}: {e}")
            continue

In [23]:
get_top_250_films

247

In [24]:
print("Количество нулевых значений в: ")
for i in result_list:
    print( i + " - " + str(result_list[i].count(None)))

Количество нулевых значений в: 
title - 0
genre - 0
year - 0
country - 0
rate - 0
description - 0


In [25]:
for key, value in result_list.items():
    print(f"{key}: {len(value)}")

title: 247
genre: 247
year: 247
country: 247
rate: 247
description: 247


In [26]:
file_name = 'Top250Films.csv'
df = pd.DataFrame(data=result_list)
df.to_csv(file_name)
df.head(20)

,title,genre,year,country,rate,description
0,Ангелы Ладоги,"Драма, История, Военный",2026,Россия,8.8,"Тихон Жизневский, Роман Евдокимов и Виктор Доб..."
1,Твоё сердце будет разбито,Мелодрама,2026,Россия,8.7,Скромница влюбляется в главного хулигана школы...
2,Новая тёща,Комедия,2026,Россия,8.4,Продолжение народной комедии с Гариком Харламо...
3,1+1,"Драма, Комедия, Биография",2011,Франция,9.0,Реальная история дружбы «принца» и «нищего». П...
4,Роднина,"Драма, Спорт, Биография",2025,Россия,8.4,Драма о легендарной фигуристке Ирине Родниной....
5,Чебурашка 2,"Фэнтези, Комедия, Приключения, Семейный",2026,Россия,8.5,Уже год Чебурашка живет у Гены. Шаловливый уша...
6,Тюльпаны,"Комедия, Мелодрама",2026,Россия,8.7,Звездный ансамбль артистов в комедийно-романти...
7,Уволить Жору (2026),Комедия,2026,Россия,8.2,Амбициозный провинциал Максим строит успешную ...
8,Несвятая Валентина,"Комедия, Мелодрама",2026,Россия,7.5,Зимняя романтическая комедия с Анастасией Крас...
9,Добрый доктор,Комедия,2026,Россия,8.5,Комедия от создателей «Полицейский с Рублевки....


In [27]:
print(f"Размер датасета: {df.shape[0]} фильмов × {df.shape[1]} полей")


Размер датасета: 247 фильмов × 6 полей


In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 247 entries, 0 to 246
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        247 non-null    object
 1   genre        247 non-null    object
 2   year         247 non-null    object
 3   country      247 non-null    object
 4   rate         247 non-null    object
 5   description  247 non-null    object
dtypes: object(6)
memory usage: 11.7+ KB


In [30]:
df['rate'].describe()

count     247
unique     30
top       8.8
freq       23
Name: rate, dtype: object

In [34]:
df['country'].describe()

count        247
unique        20
top       Россия
freq         156
Name: country, dtype: object

In [35]:
df['genre'].describe()

count         247
unique        108
top       Комедия
freq           37
Name: genre, dtype: object

In [33]:
df['rate'] = pd.to_numeric(df['rate'], errors='coerce')
display(df.nlargest(10, 'rate')[['title', 'rate', 'year', 'country']])

,title,rate,year,country
156,Шаг к мечте: Раз и навсегда,9.5,2023,Китай
232,Огонь,9.4,2020,Россия
97,Честный развод 2,9.3,2022,Россия
3,1+1,9.0,2011,Франция
12,На острие,9.0,2019,Россия
15,Первый на Олимпе,9.0,2025,Россия
86,Батя,9.0,2021,Россия
106,Эрнест и Селестина: Новые приключения,9.0,2022,Люксембург
114,Ла-Ла Ленд,9.0,2016,США
142,Белый снег,9.0,2021,Россия
